# STGAN su ERA5: risultati del modello ConvGRU

Il notebook legge solo i file che le run hanno già scritto sul server: storie di training, checkpoint, score
e log dei controlli. Non usa la GPU e non ricalcola il modello; gli score sono letti su un istante ogni
`STRIDE`, quindi le quote possono differire di poco da quelle dei log completi.

Il percorso:
1. **Fix del dropout.** Senza fix il discriminatore riconosceva la ricostruzione dal rumore del dropout del
   generatore. Con il fix vede la ricostruzione senza dropout.
2. **Run `_cleanD`.** Fix del dropout, normalizzazione min-max, nessun peso delle variabili. 6 epoche,
   training 1980–2003.
3. **Run `_ep3`.** Fix del dropout, normalizzazione stagionale con trend, pesi `daily_change`. 3 epoche,
   training 1980–2004. È la run degli score e dei controlli più avanti.
4. **Sweep in corso.** Training 1980–1999, validazione 2000–2004 senza i valori anomali. L'obiettivo è la
   PCA-MMD per epoca.

**Avvertenze.** Ogni run ha un solo seed. Gli score della run `_ep3` sono calibrati sugli anni di test
(climatologia e soglie): dopo lo sweep questa calibrazione passa sulla validazione.

In [ ]:
from pathlib import Path
import json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(os.environ['PHYSIQ_ROOT']) if os.environ.get('PHYSIQ_ROOT') else next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Apri il notebook dalla cartella del repository physiq_pv')
OUT = ROOT / 'outputs'
RUN = OUT / 'stgan_convgru_fullgrid_recent16_cyclic_bf16_cleanD_wdaily_seasonal_trend_ep3'
RUNS = {  # etichetta -> cartella della run (training_history.csv e model_epoch_*.pt)
    'prima del fix': OUT / 'stgan_convgru_fullgrid_recent16_cyclic_fp32',
    '_cleanD: fix, min-max': OUT / 'stgan_convgru_fullgrid_recent16_cyclic_bf16_cleanD',
    '_ep3: fix, stagionale, trend, pesi': RUN,
}
SWEEP_DIR = OUT / 'stgan_wandb' / 'era5'
SWEEP_LOG = OUT / 'sweep_agent.log'
NORMAL_MASKS = OUT / 'era5_sweep_1999' / 'normal_masks'
SCORES = {  # stesso periodo di test e stesse celle
    'modello': Path(f'{RUN}_gen_clim'),
    'distanza dal normale': Path(f'{RUN}_no_model_bal_clim'),
    "variazione dall'istante prima": Path(f'{RUN}_previous_bal_clim'),
}
LEAD_LOGS = ['anticipo_modello_tutti.log', 'anticipo_controlli.log']
STRIDE = 29           # un istante ogni 29: dispari, così si leggono tutte le ore del giorno
FEATURE_STRIDE = 97   # per l'errore per variabile, che ha 15 valori per cella
TOP_PERCENT = 5
CHANCE = np.log(2)    # perdita di D quando risponde al caso (con il 1/2 del codice degli autori)
SAVE_FIGURES = False  # True salva le figure in PNG
FIGURE_DIR = OUT / 'figure_risultati'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

def show(figure, name):
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        figure.savefig(FIGURE_DIR / f'{name}.png', bbox_inches='tight', dpi=150)
    plt.show()

def history(directory):
    files = sorted(Path(directory).glob('**/training_history.csv'), key=lambda path: len(path.parts))
    return pd.read_csv(files[0]) if files else None

def last_checkpoint(directory):
    files = sorted(Path(directory).glob('**/model_epoch_*.pt'),
                   key=lambda path: int(re.search(r'_epoch_(\d+)', path.name).group(1)))
    if not files:
        return None
    import torch
    return torch.load(files[-1], map_location='cpu', weights_only=False)

def find_log(name):
    return next((path for path in (ROOT / name, OUT / name) if path.is_file()), None)

## 1. Le run

Configurazione letta dall'ultimo checkpoint di ogni run. Le run dello sweep si trovano in
`outputs/stgan_wandb/era5/<id W&B>`; il loro stato (`complete` / `failed`) è quello scritto in `wandb_run.json`.

In [ ]:
def describe(label, directory, status=None):
    payload = last_checkpoint(directory)
    table = history(directory)
    row = {'run': label, 'epoche': None if table is None else int(table['epoch'].max())}
    if payload is not None:
        rates = payload.get('learning_rates') or {}
        model = payload.get('model_config') or {}
        row.update({
            'lr G': rates.get('generator_learning_rate'), 'lr D': rates.get('discriminator_learning_rate'),
            'D:G': payload.get('discriminator_generator_update_ratio', '1:1'),
            'peso ricostruzione': payload.get('generator_reconstruction_weight'),
            'normalizzazione': (payload.get('normalization') or {}).get('kind'),
            'pesi variabili': 'no' if model.get('reconstruction_feature_weights') is None else 'sì',
            'fix dropout': 'sì' if model.get('discriminator_input_dropout', True) is False else 'no',
            'training': (payload.get('splits') or {}).get('train'),
        })
    if status:
        row['stato'] = status
    return row

rows = [describe(label, directory) for label, directory in RUNS.items() if directory.is_dir()]
SWEEP = {}  # id W&B -> cartella dei risultati
for identity_file in sorted(SWEEP_DIR.glob('*/wandb_run.json')):
    identity = json.loads(identity_file.read_text(encoding='utf-8'))
    SWEEP[identity['id']] = identity_file.parent / 'results'
    rows.append(describe(f"sweep {identity['id']}", identity_file.parent / 'results', identity.get('status')))
runs_table = pd.DataFrame(rows)
display(runs_table)
missing = [label for label, directory in RUNS.items() if not directory.is_dir()]
if missing:
    print('Cartelle non trovate:', missing)

## 2. Il fix del dropout

Il discriminatore D dà la probabilità che un campo sia **generato**: l'obiettivo è 0 sui veri e 1 sui
generati. Se D non distingue, la sua perdita vale ln 2 ≈ 0,693 (linea tratteggiata) e le due uscite medie
valgono 0,5. Prima del fix D vedeva la ricostruzione con il dropout del generatore. Con il fix vede la
ricostruzione senza dropout.

In [ ]:
pair = {label: history(RUNS[label]) for label in ('prima del fix', '_cleanD: fix, min-max')}
pair = {label: table for label, table in pair.items() if table is not None}
if pair:
    figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for label, table in pair.items():
        axes[0].plot(table['epoch'], table['discriminator_loss'], marker='o', label=label)
        if {'discriminator_real_mean', 'discriminator_fake_mean'} <= set(table.columns):
            line, = axes[1].plot(table['epoch'], table['discriminator_fake_mean'], marker='o', label=f'{label}: generati')
            axes[1].plot(table['epoch'], table['discriminator_real_mean'], marker='s', linestyle='--',
                         color=line.get_color(), label=f'{label}: veri')
    axes[0].axhline(CHANCE, color='grey', linestyle=':', label='caso (ln 2)')
    axes[0].set(title='Perdita di D per epoca', xlabel='Epoca', ylabel='BCE media')
    axes[1].axhline(0.5, color='grey', linestyle=':')
    axes[1].set(title='Uscita media di D (probabilità di "generato")', xlabel='Epoca', ylim=(0, 1))
    axes[0].legend()
    if axes[1].get_legend_handles_labels()[0]:
        axes[1].legend(fontsize=8)
    show(figure, 'fix_dropout')
    for label, table in pair.items():
        print(label)
        display(table[[c for c in ('epoch', 'discriminator_loss', 'discriminator_real_mean',
                                   'discriminator_fake_mean', 'generator_loss') if c in table]].round(4))
else:
    print('Storie di training non trovate')

## 3. Le perdite epoca per epoca

Le perdite di D e il termine avversario di G sono BCE e si possono confrontare tra le run. L'errore di
ricostruzione invece no: cambiano normalizzazione e pesi delle variabili. Per questo è diviso per il suo
valore all'epoca 1, e il grafico mostra solo come cambia nel tempo.

In [ ]:
tables = {label: history(directory) for label, directory in RUNS.items() if label != 'prima del fix'}
tables.update({f'sweep {run_id}': history(directory) for run_id, directory in SWEEP.items()})
tables = {label: table for label, table in tables.items() if table is not None and len(table)}
figure, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for label, table in tables.items():
    axes[0].plot(table['epoch'], table['discriminator_loss'], marker='o', label=label)
    if 'generator_adversarial_loss' in table:
        axes[1].plot(table['epoch'], table['generator_adversarial_loss'], marker='o', label=label)
    if 'generator_reconstruction_loss' in table:
        reconstruction = table['generator_reconstruction_loss']
        axes[2].plot(table['epoch'], reconstruction / reconstruction.iloc[0], marker='o', label=label)
axes[0].axhline(CHANCE, color='grey', linestyle=':')
axes[0].set(title='Perdita di D', xlabel='Epoca')
axes[1].set(title='Termine avversario di G', xlabel='Epoca')
axes[2].axhline(1, color='grey', linestyle=':')
axes[2].set(title="Errore di ricostruzione / epoca 1", xlabel='Epoca')
axes[0].legend(fontsize=8)
show(figure, 'perdite_per_epoca')

## 4. Dentro le epoche: le righe di avanzamento dello sweep

Ogni punto è la media di un intervallo di batch (2.919, cioè 1/20 di epoca) letta dal log dell'agente dello
sweep. Una × segna la fine di una run fermata da una perdita non finita (`FloatingPointError`): si vede se
prima c'era una deriva o se il valore è arrivato all'improvviso.

In [ ]:
PROGRESS = re.compile(r'epoch=(\d+)/(\d+) batch=(\d+)/(\d+) D_mean=([\d.eE+-]+) G_mean=([\d.eE+-]+)')
progress, crashed, current = {}, set(), None
if SWEEP_LOG.is_file():
    for line in SWEEP_LOG.read_text(encoding='utf-8', errors='replace').splitlines():
        match = re.search(r'Syncing run (\S+)', line)
        if match:
            current = match.group(1)
        match = PROGRESS.search(line)
        if match and current:
            epoch, _, batch, total, d_mean, g_mean = match.groups()
            if batch == total:  # the last line of an epoch covers only the few batches left over
                continue
            progress.setdefault(current, []).append(
                (int(epoch) - 1 + int(batch) / int(total), float(d_mean), float(g_mean)))
        if 'FloatingPointError' in line and current:
            crashed.add(current)
if progress:
    figure, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
    for name, points in progress.items():
        position, d_mean, g_mean = (np.array([point[index] for point in points]) for index in range(3))
        line, = axes[0].plot(position, d_mean, label=name)
        axes[1].plot(position, g_mean, color=line.get_color(), label=name)
        if name in crashed:
            for axis, values in zip(axes, (d_mean, g_mean)):
                axis.scatter(position[-1], values[-1], marker='x', s=90, color='red', zorder=4)
    axes[0].axhline(CHANCE, color='grey', linestyle=':')
    axes[0].set(title='Perdita di D (media per intervallo)', xlabel='Epoca')
    axes[1].set(title='Perdita di G (media per intervallo)', xlabel='Epoca')
    axes[0].legend(fontsize=8)
    show(figure, 'sweep_avanzamento')
    print('Run fermate da una perdita non finita:', sorted(crashed) or 'nessuna')
else:
    print('Log dello sweep non trovato:', SWEEP_LOG)

## 5. Lo sweep: PCA-MMD di validazione per epoca

L'obiettivo dello sweep confronta i campi di validazione normali (2000–2004) con le loro ricostruzioni,
nello spazio della PCA stimata sul training (300 componenti). Più è basso, più le ricostruzioni somigliano al
meteo vero. Il voto di una run è la media delle ultime 5 epoche su 6 (linea tratteggiata: media mobile). Le
barre sono la deviazione standard sui 5 sottoinsiemi di validazione.

In [ ]:
mmd = {run_id: history(directory) for run_id, directory in SWEEP.items()}
mmd = {run_id: table for run_id, table in mmd.items()
       if table is not None and 'validation_pca_mmd_mean' in table}
if mmd:
    figure, axis = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
    configuration = runs_table.set_index('run')
    for run_id, table in mmd.items():
        row = configuration.loc[f'sweep {run_id}'] if f'sweep {run_id}' in configuration.index else None
        label = run_id if row is None else (f"{run_id}: lr G {row['lr G']:.2g}, D/G {row['lr D'] / row['lr G']:.2g}, "
                                            f"{row['D:G']}, peso {row['peso ricostruzione']:.0f}")
        spread = np.sqrt(table['validation_pca_mmd_variance']) if 'validation_pca_mmd_variance' in table else None
        line = axis.errorbar(table['epoch'], table['validation_pca_mmd_mean'], yerr=spread, marker='o',
                             capsize=3, label=label)[0]
        axis.plot(table['epoch'], table['validation_pca_mmd_rolling_mean'], linestyle='--', color=line.get_color())
    axis.set(xlabel='Epoca', ylabel='MMD² (PCA)', title='Validazione dello sweep, epoca per epoca')
    axis.legend(fontsize=8)
    show(figure, 'sweep_mmd')
    for run_id, table in mmd.items():
        print(run_id)
        display(table[[c for c in ('epoch', 'validation_pca_mmd_mean', 'validation_pca_mmd_variance',
                                   'validation_pca_mmd_rolling_mean', 'discriminator_loss') if c in table]]
                .style.format(precision=6))
else:
    print('Nessuna run dello sweep con la PCA-MMD in', SWEEP_DIR)

### Quanti valori di validazione sono stati tolti

Un valore (cella × istante) è segnato come anomalo, senza modello, se è nel 5% più alto della distanza dal
normale stagionale **oppure** della variazione dall'istante prima. Soglie e standardizzazione sono stimate sul
training. Nella validazione i valori segnati non entrano nelle statistiche e negli input sono sostituiti dal
normale stagionale.

In [ ]:
if (NORMAL_MASKS / 'validation_mask.npy').is_file():
    mask = np.load(NORMAL_MASKS / 'validation_mask.npy', mmap_mode='r')
    train_share = np.load(NORMAL_MASKS / 'train_share.npy')
    metadata = json.loads((NORMAL_MASKS / 'metadata.json').read_text(encoding='utf-8'))
    validation_share = float(sum(np.asarray(mask[start:start + 1024]).sum()
                                 for start in range(0, len(mask), 1024)) / mask.size)
    display(pd.DataFrame({'quota di valori segnati': {
        'training': float(np.nanmean(train_share)), 'validazione': validation_share}}).style.format('{:.2%}'))
    print('Soglie (sul training):', metadata.get('levels'))
else:
    print('Maschere non trovate in', NORMAL_MASKS)

## 6. Gli score sul test (run `_ep3`)

Tre score sugli stessi anni e sulle stesse celle, dopo la climatologia:
- **modello**: errore di ricostruzione del generatore, la media dei campioni MC Dropout;
- **distanza dal normale**: quanto il valore si allontana dal normale stagionale, senza modello;
- **variazione dall'istante prima**: quanto il valore cambia in 3 ore, senza modello.

Un valore è segnato da uno score se è nel suo `TOP_PERCENT` più alto.

In [ ]:
def load_score(directory):
    manifest = json.loads((directory / 'metadata.json').read_text(encoding='utf-8'))
    file = manifest.get('anomaly_mean_file', manifest.get('scores_file', 'scores/anomaly_mean.npy'))
    scores = np.load(directory / file, mmap_mode='r')
    sample = np.array(scores[::STRIDE], dtype=np.float32)
    times = pd.to_datetime(np.load(directory / 'test_timestamps.npy'))[::STRIDE]
    return sample, times

score, times = {}, None
for name, directory in SCORES.items():
    if (directory / 'metadata.json').is_file():
        score[name], read_times = load_score(directory)
        if times is None:
            times = read_times
        elif not times.equals(read_times) or score[name].shape != next(iter(score.values())).shape:
            raise ValueError(f'{name}: non è sugli stessi istanti e celle degli altri score')
    else:
        print('Score non trovato:', directory)
if score:
    finite = np.logical_and.reduce([np.isfinite(values) for values in score.values()])
    limit = {name: float(np.percentile(values[finite], 100 - TOP_PERCENT)) for name, values in score.items()}
    flagged = {name: (values > limit[name]) & finite for name, values in score.items()}
    print(f'{len(times)} istanti letti ({times[0].date()} – {times[-1].date()}), {finite.shape[1]} celle')
    figure, axes = plt.subplots(1, len(score), figsize=(5 * len(score), 3.8), constrained_layout=True)
    for axis, (name, values) in zip(np.atleast_1d(axes), score.items()):
        data = values[finite]
        low, high = np.percentile(data, [0.1, 99.99])
        axis.hist(data, bins=200, range=(low, high), color='tab:blue', alpha=0.8)
        axis.axvline(limit[name], color='red', linestyle='--', label=f'soglia {TOP_PERCENT}%')
        axis.set(yscale='log', title=name, xlabel='score', ylabel='valori (scala log)')
        axis.legend()
    show(figure, 'istogrammi_score')

### Cosa segnala il modello che gli score semplici non segnalano

Tra i valori segnati dal modello: quanti sono segnati anche dalla distanza dal normale, dalla variazione
dall'istante prima, da almeno uno dei due, da nessuno dei due. La parte "da nessuno" è quella che il modello
aggiunge.

In [ ]:
if len(score) == 3:
    model, normal, change = (flagged[name] for name in SCORES)
    count = model.sum()
    overlap = pd.Series({
        'anche distanza dal normale': (model & normal).sum() / count,
        "anche variazione dall'istante prima": (model & change).sum() / count,
        'almeno uno dei due': (model & (normal | change)).sum() / count,
        'nessuno dei due (solo modello)': (model & ~normal & ~change).sum() / count,
    })
    display(overlap.to_frame('quota dei valori segnati dal modello').style.format('{:.1%}'))
    figure, axis = plt.subplots(figsize=(8, 3), constrained_layout=True)
    axis.barh(overlap.index[::-1], 100 * overlap.values[::-1], color=['tab:red', 'tab:green', 'tab:orange', 'tab:blue'])
    axis.set(xlabel='% dei valori segnati dal modello', xlim=(0, 100))
    show(figure, 'sovrapposizione')

    month = times.month.to_numpy()
    by_month = pd.DataFrame({name: [values[month == m].mean() if (month == m).any() else np.nan for m in range(1, 13)]
                             for name, values in flagged.items()},
                            index=range(1, 13))
    figure, axis = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
    (100 * by_month).plot(ax=axis, marker='o')
    axis.axhline(TOP_PERCENT, color='grey', linestyle=':')
    axis.set(xlabel='Mese', ylabel='% di valori segnati', title='Quando segnala ciascuno score')
    show(figure, 'segnalati_per_mese')

### Dove segnala

Quota di istanti segnati in ogni cella, con la stessa scala di colori per tutte le mappe. Per i tre score la
media sulle celle vale `TOP_PERCENT`.

In [ ]:
metadata = json.loads((RUN / 'metadata.json').read_text(encoding='utf-8')) if (RUN / 'metadata.json').is_file() else {}
grid = metadata.get('grid')
if grid and score:
    latitudes, longitudes = np.asarray(grid['latitudes']), np.asarray(grid['longitudes'])
    rows, cols = np.asarray(grid['rows']), np.asarray(grid['cols'])
    half = float(grid.get('spacing', 0.5)) / 2
    extent = (longitudes[0] - half, longitudes[-1] + half, latitudes[-1] - half, latitudes[0] + half)
    maps = {name: values.mean(axis=0) for name, values in flagged.items()}
    if len(score) == 3:
        model, normal, change = (flagged[name] for name in SCORES)
        maps['solo modello'] = (model & ~normal & ~change).mean(axis=0)
    top = np.nanpercentile(100 * np.concatenate(list(maps.values())), 99)  # stessa scala per tutte le mappe
    figure, axes = plt.subplots(1, len(maps), figsize=(4.6 * len(maps), 4), constrained_layout=True)
    for axis, (name, share) in zip(np.atleast_1d(axes), maps.items()):
        image = np.full((len(latitudes), len(longitudes)), np.nan)
        image[rows, cols] = 100 * share
        shown = axis.imshow(image, extent=extent, origin='upper', cmap='viridis',
                            vmin=0, vmax=top)
        axis.set(title=name, xlabel='lon', ylabel='lat')
        axis.grid(False)
        figure.colorbar(shown, ax=axis, shrink=0.8, label='% istanti segnati')
    show(figure, 'mappe')

### Di cosa è fatto l'errore del modello

L'errore quadratico per variabile salvato dallo scoring (`scores/feature_scores.npy`), con i pesi delle
variabili della run, come lo vede la perdita del modello. Quota di ogni variabile su tutto l'errore e sui
valori segnati dal modello.

In [ ]:
feature_file = RUN / 'scores' / 'feature_scores.npy'
backend = metadata.get('backend', {})
weights = backend.get('reconstruction_feature_weights')
names = (metadata.get('preparation') or {}).get('features') or (list(weights) if weights else None)
if feature_file.is_file() and 'modello' in score and names:
    errors = np.load(feature_file, mmap_mode='r')
    model_scores = np.load(SCORES['modello'] / json.loads((SCORES['modello'] / 'metadata.json').read_text(
        encoding='utf-8')).get('anomaly_mean_file', 'scores/anomaly_mean.npy'), mmap_mode='r')
    if errors.shape[:2] != model_scores.shape:
        raise ValueError('Errori per variabile e score del modello non hanno la stessa forma')
    weight = np.array([weights[name] for name in names]) if weights else np.ones(len(names))
    total, top = np.zeros(len(names)), np.zeros(len(names))
    rows_read = np.arange(0, len(errors), FEATURE_STRIDE)
    for start in range(0, len(rows_read), 16):
        chosen = rows_read[start:start + 16]
        block = np.asarray(errors[chosen], dtype=np.float64) * weight
        values = np.asarray(model_scores[chosen], dtype=np.float64)
        valid = np.isfinite(values) & np.isfinite(block).all(axis=-1)
        total += block[valid].sum(axis=0)
        top += block[valid & (values > limit['modello'])].sum(axis=0)
    shares = pd.DataFrame({'tutto l\'errore': total / total.sum(), f'valori segnati ({TOP_PERCENT}%)': top / top.sum()},
                          index=names).sort_values('tutto l\'errore')
    figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
    (100 * shares).plot.barh(ax=axis)
    axis.set(xlabel="% dell'errore pesato", title="Errore del modello per variabile")
    show(figure, 'errore_per_variabile')
    display(shares.sort_values('tutto l\'errore', ascending=False).style.format('{:.1%}'))
else:
    print('Errori per variabile non disponibili:', feature_file)

## 7. Anticipo: chi segnala per primo l'inizio di un episodio

Tabelle di `scripts/check_era5_lead.py` (log `anticipo_*.log`, run `_ep3`, soglie sul test). Per ogni inizio di
un episodio dello score della riga, nella stessa cella: quota di casi in cui l'altro score **segnalava già**
(`already`), e da quante ore. Le righe `chance` sono lo stesso conteggio in istanti in cui lo score della riga
è quieto e non comincia nulla: sono il riferimento del caso.

In [ ]:
COLUMNS = ['starts', 'already', '3h', '6h', '9-12h', '15-24h', '>24h', 'lead med h', 'gap', 'together',
           'later', 'lag med h', 'none']
blocks = []
for name in LEAD_LOGS:
    path = find_log(name)
    if path is None:
        print('Log non trovato:', name)
        continue
    header = None
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        if line.startswith('[lead]'):
            header = line
            blocks.append({'log': name, 'header': line, 'rows': {}})
        elif header and line.startswith(('starts of', '  chance')):
            values = line[34:].split()
            if len(values) == len(COLUMNS):
                blocks[-1]['rows'][line[:34].strip()] = [float(value) for value in values]
summary = []
for block in blocks:
    table = pd.DataFrame(block['rows'], index=COLUMNS).T
    print(block['log'], '|', block['header'].split('|')[1].strip())
    display(table.round(1))
    starts = [label for label in table.index if label.startswith('starts of')]
    if starts:
        label = starts[0]
        chance = next((row for row in table.index if row.startswith('chance')), None)
        summary.append({'confronto': f"{block['header'].split('|')[1].strip()}",
                        'già acceso': table.loc[label, 'already'],
                        'già acceso da ≥6 h': table.loc[label, ['6h', '9-12h', '15-24h', '>24h']].sum(),
                        'caso': table.loc[chance, 'already'] if chance else np.nan})
if summary:
    summary = pd.DataFrame(summary)
    figure, axis = plt.subplots(figsize=(10, 0.6 * len(summary) + 1.5), constrained_layout=True)
    position = np.arange(len(summary))
    axis.barh(position - 0.27, summary['già acceso'], height=0.27, label='già acceso')
    axis.barh(position, summary['già acceso da ≥6 h'], height=0.27, label='già acceso da ≥6 h')
    axis.barh(position + 0.27, summary['caso'], height=0.27, label='caso', color='grey')
    axis.set_yticks(position, summary['confronto'], fontsize=7)
    axis.set(xlabel='% degli inizi di episodio (prima riga di ogni tabella)')
    axis.invert_yaxis(); axis.legend()
    show(figure, 'anticipo')

## 8. In sintesi

- **Fix del dropout.** Senza fix, D riconosce la ricostruzione dal rumore del dropout e non guarda il meteo.
  Con il fix la perdita di D resta sotto il caso ma non lontana: D distingue poco, coerente con quanto
  descrivono gli autori di STGAN a fine training.
- **Dopo l'epoca 3.** Nella `_cleanD` (6 epoche) e nella prima run dello sweep il modello migliora fino
  all'epoca 2–3. Poi D separa un po' di più e la ricostruzione (o la MMD) peggiora. Lo sweep dirà se dipende
  dai parametri della GAN o se è una tendenza del modello.
- **Cosa aggiunge il modello.** Una parte dei suoi valori segnati non è segnata da nessuno dei due score
  semplici (sezione 6). Sulle perturbazioni rapide forti il modello è spesso già acceso all'inizio, più degli
  score semplici e molto più del caso (sezione 7). Sulle anomalie lente il normale stagionale resta migliore.
- **Limiti.** Un solo seed per run. Gli score `_ep3` sono calibrati sugli anni di test. È un anticipo
  rispetto agli altri rilevatori, non una previsione.